In [1]:
import datetime
## track delay graph
import pandas as pd
from pandas import DataFrame
import numpy as np
import networkx as nx
import peartree as pear
import partridge as ptg
import matplotlib.pyplot as plt
from itertools import tee, product
import warnings
import matplotlib.colors
import re
from functools import reduce
import json
from typing import Literal
from datetime import datetime, date
warnings.filterwarnings('ignore')
matplotlib.style.use('default')
import xarray as xr
np.set_printoptions(suppress=True)

def iterator(L):
	a, b = tee(L)
	next(b, None)
	return list(zip(a,b))


In [2]:
def do_all():

	def from_mta_time(t):
		return(t/100)*60
	lines = sorted(['A','C','E','B','D','F','FX','M','G','J','Z','L','N','Q','R','W','FS','GS','H','1','2','3','4','5','6','6X','7','7X'])
	def midday_deltas(month, day):
		mtatime = [9.5*60*100, 15.5*60*100]
		expected_trips = pd.read_csv('gtfs_subway/trips.txt')
		actual_trips = pd.read_csv(f"sept27data/subwaydatanyc_2025-{month}-{day}_trips.csv")

		expected_times = pd.read_csv('gtfs_subway/stop_times.txt')
		actual_times = pd.read_csv(f"sept27data/subwaydatanyc_2025-{month}-{day}_stop_times.csv")


		def format_time(t, month):
			'''Helper to format MTA timestamps that go over 24 hours due to delays'''
			rd = [int(n) for n in t.split(':')]
			d = 0
			if rd[0] >= 24:
				rd[0] = rd[0] % 24
				d = 1 
			s = ':'.join([str(k) for k in rd])
			datestr = datetime.strptime(f"{month}/{day+d}/2025 {s}", "%m/%d/%Y %H:%M:%S")
			return datestr


		actual_trips['init_time'] = actual_trips.trip_id.map(lambda s: int(s.split('_')[0]))
		actual_trips = actual_trips[actual_trips.init_time.between(mtatime[0],mtatime[1])]
		actual_trips = actual_trips[actual_trips.route_id != 'SI']
		actual_trips.sort_values('init_time', inplace=True)
		actual_trips['short_id'] = actual_trips.trip_id.map(lambda s: ''.join(re.split(r"(\.{1,2}[SN])",s)[:-1]))
		expected_trips = expected_trips[~expected_trips.route_id.isin(['SI','SS'])]
		expected_trips = expected_trips[expected_trips.service_id == 'Weekday']
		expected_trips['short_id'] = expected_trips.trip_id.map(lambda s: '_'.join(s.split('_')[-2:]))
		expected_trips['init_time'] = expected_trips.trip_id.map(lambda s: int(s.split('_')[-2]))
		expected_trips = expected_trips[expected_trips.init_time.between(mtatime[0],mtatime[1])]
		expected_trips.short_id = expected_trips.short_id.map(lambda s: s[:-3])
		expected_trips = expected_trips.astype(str)
		expected_trips = expected_trips[['route_id','short_id', 'trip_id']]
		expected_trips.columns = ['route_id', 'short_id', 'full_trip_id']
		actual_trips = actual_trips[['trip_uid','route_id','trip_id','direction_id','start_time','short_id','init_time']]
		merged = pd.merge(actual_trips, expected_trips, on='short_id')
		merged['unique'] = merged.trip_id.map(lambda s: merged.trip_id.value_counts(sort=False).to_dict()[s])
		merged['route_id'] = list(zip(merged['route_id_x'], merged['route_id_y']))
		merged.route_id = merged.route_id.map(lambda x: x[0] if len(np.unique_values(list(x)))==1 else x)
		idkey = {v: k for k,v in zip(merged.trip_uid, merged.full_trip_id)}
		expected_times = expected_times[~expected_times.trip_id.str.startswith('SIR-')]
		expected_times['id_key'] =expected_times.trip_id.map(lambda s: '_'.join(s.split('-')[-1].split('_')[1:]))
		expected_times['key'] = expected_times.trip_id.map(lambda k: idkey.get(k))+expected_times.stop_id.astype(str)
		expected_times = expected_times[~expected_times.key.isna()]
		expected_times.sort_values('key')
		expected_times.columns = ['full_trip_id', 'stop_id','expected_arrival','departure','seq','short_id','key']
		real_time = actual_times.copy()
		real_time = real_time[real_time['trip_uid'].isin(merged.trip_uid)]
		real_time.stop_id = real_time.stop_id.map(lambda a: str(a[:-1]))
		real_time = real_time[~(real_time.arrival_time.isna() & real_time.departure_time.isna())]
		real_time['key'] = [a+b for a,b in zip(real_time.trip_uid, real_time.stop_id)]
		real_time.arrival_time = [int(a[np.nanargmax(a)]) for a in zip(real_time.arrival_time, real_time.departure_time)]
		real_time['init_time'] = real_time.trip_uid.map(lambda s: int(s.split('_')[0]))
		real_time.arrival_time = real_time.arrival_time.map(lambda t: datetime.fromtimestamp(t).time())
		real_time.sort_values('key')
		ddict = dict(zip(expected_trips.full_trip_id.values,expected_trips.short_id.map(lambda n: n[-1])))
		iddict = dict(zip(expected_trips.full_trip_id, expected_trips.route_id))
		expected_times['route'] = expected_times.full_trip_id.map(lambda x: iddict.get(x))
		expected_times['direction'] = expected_times.full_trip_id.map(lambda x: ddict.get(x))
		full_merge =pd.merge(real_time, expected_times, on='key')
		full_merge = full_merge[full_merge.expected_arrival.notna()]
		full_merge = full_merge[['stop_id_x','arrival_time','expected_arrival','seq', 'route','direction']]
		full_merge.drop('seq', axis=1,inplace=True)
		full_merge['delta']=[(format_time(a)-format_time(b)).total_seconds() for a,b in zip(full_merge.arrival_time.astype(str), full_merge.expected_arrival.astype(str))]
		deltas = full_merge[['stop_id_x','route','direction','delta']]
		return deltas

In [3]:
def midday_deltas(day):
	mtatime = [9.5*60*100, 15.5*60*100]
	expected_trips = pd.read_csv('gtfs_subway/trips.txt')
	actual_trips = pd.read_csv(f"sept27data/subwaydatanyc_2025-09-{day}_trips.csv")

	expected_times = pd.read_csv('gtfs_subway/stop_times.txt')
	actual_times = pd.read_csv(f"sept27data/subwaydatanyc_2025-09-{day}_stop_times.csv")


	def format_time(t):
		'''Helper to format MTA timestamps that go over 24 hours due to delays'''
		rd = [int(n) for n in t.split(':')]
		d = 0
		if rd[0] >= 24:
			rd[0] = rd[0] % 24
			d = 1 
		s = ':'.join([str(k) for k in rd])
		datestr = datetime.strptime(f"9/{day+d}/2025 {s}", "%m/%d/%Y %H:%M:%S")
		return datestr


	actual_trips['init_time'] = actual_trips.trip_id.map(lambda s: int(s.split('_')[0]))
	actual_trips = actual_trips[actual_trips.init_time.between(mtatime[0],mtatime[1])]
	actual_trips = actual_trips[actual_trips.route_id != 'SI']
	actual_trips.sort_values('init_time', inplace=True)
	actual_trips['short_id'] = actual_trips.trip_id.map(lambda s: ''.join(re.split(r"(\.{1,2}[SN])",s)[:-1]))
	expected_trips = expected_trips[~expected_trips.route_id.isin(['SI','SS'])]
	expected_trips = expected_trips[expected_trips.service_id == 'Weekday']
	expected_trips['short_id'] = expected_trips.trip_id.map(lambda s: '_'.join(s.split('_')[-2:]))
	expected_trips['init_time'] = expected_trips.trip_id.map(lambda s: int(s.split('_')[-2]))
	expected_trips = expected_trips[expected_trips.init_time.between(mtatime[0],mtatime[1])]
	expected_trips.short_id = expected_trips.short_id.map(lambda s: s[:-3])
	expected_trips = expected_trips.astype(str)
	expected_trips = expected_trips[['route_id','short_id', 'trip_id']]
	expected_trips.columns = ['route_id', 'short_id', 'full_trip_id']
	actual_trips = actual_trips[['trip_uid','route_id','trip_id','direction_id','start_time','short_id','init_time']]
	merged = pd.merge(actual_trips, expected_trips, on='short_id')
	merged['unique'] = merged.trip_id.map(lambda s: merged.trip_id.value_counts(sort=False).to_dict()[s])
	merged['route_id'] = list(zip(merged['route_id_x'], merged['route_id_y']))
	merged.route_id = merged.route_id.map(lambda x: x[0] if len(np.unique_values(list(x)))==1 else x)
	idkey = {v: k for k,v in zip(merged.trip_uid, merged.full_trip_id)}
	expected_times = expected_times[~expected_times.trip_id.str.startswith('SIR-')]
	expected_times['id_key'] =expected_times.trip_id.map(lambda s: '_'.join(s.split('-')[-1].split('_')[1:]))
	expected_times['key'] = expected_times.trip_id.map(lambda k: idkey.get(k))+expected_times.stop_id.astype(str)
	expected_times = expected_times[~expected_times.key.isna()]
	expected_times.sort_values('key')
	expected_times.columns = ['full_trip_id', 'stop_id','expected_arrival','departure','seq','short_id','key']
	real_time = actual_times.copy()
	real_time = real_time[real_time['trip_uid'].isin(merged.trip_uid)]
	real_time.stop_id = real_time.stop_id.map(lambda a: str(a[:-1]))
	real_time = real_time[~(real_time.arrival_time.isna() & real_time.departure_time.isna())]
	real_time['key'] = [a+b for a,b in zip(real_time.trip_uid, real_time.stop_id)]
	real_time.arrival_time = [int(a[np.nanargmax(a)]) for a in zip(real_time.arrival_time, real_time.departure_time)]
	real_time['init_time'] = real_time.trip_uid.map(lambda s: int(s.split('_')[0]))
	real_time.arrival_time = real_time.arrival_time.map(lambda t: datetime.fromtimestamp(t).time())
	real_time.sort_values('key')
	ddict = dict(zip(expected_trips.full_trip_id.values,expected_trips.short_id.map(lambda n: n[-1])))
	iddict = dict(zip(expected_trips.full_trip_id, expected_trips.route_id))
	expected_times['route'] = expected_times.full_trip_id.map(lambda x: iddict.get(x))
	expected_times['direction'] = expected_times.full_trip_id.map(lambda x: ddict.get(x))
	full_merge =pd.merge(real_time, expected_times, on='key')
	full_merge = full_merge[full_merge.expected_arrival.notna()]
	full_merge = full_merge[['stop_id_x','arrival_time','expected_arrival','seq', 'route','direction']]
	full_merge.drop('seq', axis=1,inplace=True)
	full_merge['delta']=[(format_time(a)-format_time(b)).total_seconds() for a,b in zip(full_merge.arrival_time.astype(str), full_merge.expected_arrival.astype(str))]
	deltas = full_merge[['stop_id_x','route','direction','delta']]
	return deltas

In [4]:
sep22 = midday_deltas(22)
sep23 = midday_deltas(23)
sep24 = midday_deltas(24)
sep25 = midday_deltas(25)
sep26 = midday_deltas(26)
sep27week = pd.concat([sep22,sep23,sep24,sep25,sep26]).reset_index(drop=True)
sep27week.columns = ['stop','line','dir','delta']
sep27avg = sep27week.groupby(["stop",'dir','line']).agg({'delta':'mean'})
deltadict=dict(zip(sep27avg.index.values, sep27avg.delta))
mtidx = pd.MultiIndex.from_tuples(list(product(all_stops, ['N','S'])), names=['stop','dir'])
full_grid = pd.DataFrame(index=mtidx, columns=lines)
for k,v in deltadict.items():
	full_grid.at[(k[0],k[1]),k[2]] = v


NameError: name 'all_stops' is not defined

In [ ]:
full_grid[full_grid.isna().all(1)]

,,1,2,3,4,5,6,6X,7,7X,A,...,GS,H,J,L,M,N,Q,R,W,Z
stop,dir,,,,,,,,,,,,,,,,,,,,,
711,S,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
713,S,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
H01,S,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
## we can extrapolate 711 and 713 Southbound since they dont share tracks and only use the 7, so the average delay from previous and following stops will work here
full_grid.at[('711','S'),'7'] = (full_grid.loc['710','S']['7']+full_grid.loc['712','S']['7'])/2
full_grid.at[('713','S'),'7'] = (full_grid.loc['712','S']['7']+full_grid.loc['714','S']['7'])/2

## aqueduct racetrack is a oneway stop so we can just assume the delay would be the same
full_grid.at[('H01','S'),'A'] = full_grid.at[('H01','N'),'A']
full_grid[full_grid.isna().all(1)]
full_grid_md = full_grid.copy()
aggstops = {k: lambda x: np.nanmean(x, axis=0, dtype=np.float64) for k in lines}
aggstops['stop'] = 'first'
full_grid_md_avg = full_grid_md.reset_index().drop('dir', axis=1).groupby('stop').agg(aggstops)
delay_times = full_grid_md_avg.drop('stop',axis=1).apply(lambda x: np.nanmean(x),axis=1)
np.round(delay_times.values.clip(0)/60).clip(0,5)
daymindf = pd.DataFrame(delay_times, columns=['seconds'])
daymindf['mins'] = np.round(delay_times.values.clip(0)/60).clip(0,5)
daymindf.mins.to_csv('data/delay_minutes_midday.csv')

In [ ]:
#raise RuntimeError('i forgot variable names in notebooks are cringe')

### get tph

In [ ]:
import time

In [ ]:
expected_trips = pd.read_csv('gtfs_subway/trips.txt')
expected_times = pd.read_csv('gtfs_subway/stop_times.txt')
expected_trips = expected_trips[expected_trips.service_id == 'Weekday']
expected_trips = expected_trips[expected_trips.route_id != 'SI']
expected_times = expected_times[expected_times.trip_id.isin(expected_trips.trip_id)]
routedict = dict(zip(expected_trips.trip_id, expected_trips.route_id))
dirdict = dict(zip(expected_trips.trip_id, expected_trips.direction_id))
expected_times['dir'] = expected_times.trip_id.map(lambda id: dirdict.get(id))
expected_times['route_id'] = expected_times.trip_id.map(lambda id: routedict.get(id))
expected_times['hour'] = expected_times.arrival_time.map(lambda h: float(h[:4].replace(':','.')))
expected_times = expected_times[expected_times.hour.between(11,14,'both')]
tph = expected_times.copy()[['stop_id','dir','route_id']]
tph.dir = tph.dir.map(lambda d: "NS"[d])
tph = tph.groupby(['stop_id','dir']).agg({'route_id':lambda w: w.to_list()})

tph['tph'] = tph.route_id.map(lambda w: (pd.value_counts(w)/3).to_dict()).values
tphavg = tph.drop('route_id', axis=1)
tphavg['route'] = tph.tph.map(lambda w: list(w.keys()))
tphavg['tphs'] = tph.tph.map(lambda w: list(w.values()))
tphavg.drop('tph',axis=1,inplace=True)
tphavg = tphavg.reset_index().explode(['route','tphs'])
tphavg.tphs = tphavg.tphs.astype(float)
tphavg = tphavg.groupby(['stop_id','route']).agg({'tphs':'mean'})
tphavg.reset_index(inplace=True)
tphavg.stop_id = tphavg.stop_id.astype(str)
tphavg = tphavg.sort_values('stop_id').groupby('stop_id').agg(lambda x: x.to_list())
tphavg.reset_index(inplace=True)
tphdict = dict(zip(tphavg.stop_id, [dict(zip(a,b)) for a,b in zip(tphavg.route, tphavg.tphs)]))

In [ ]:
tphdict

{'101': {'1': 11.166666666666666},
 '103': {'1': 11.0},
 '104': {'1': 11.0},
 '106': {'1': 10.833333333333334},
 '107': {'1': 11.166666666666666},
 '108': {'1': 11.0},
 '109': {'1': 11.0},
 '110': {'1': 11.0},
 '111': {'1': 11.0},
 '112': {'1': 11.0},
 '113': {'1': 11.0},
 '114': {'1': 11.0},
 '115': {'1': 11.0},
 '116': {'1': 11.0},
 '117': {'1': 10.833333333333332},
 '118': {'1': 11.0},
 '119': {'1': 10.833333333333334},
 '120': {'1': 11.0, '2': 8.0, '3': 7.833333333333334},
 '121': {'1': 10.833333333333332},
 '122': {'1': 11.0},
 '123': {'2': 8.0, '3': 8.166666666666668, '1': 11.0},
 '124': {'1': 11.0},
 '125': {'1': 10.833333333333332},
 '126': {'1': 11.166666666666668},
 '127': {'1': 10.833333333333332, '2': 8.166666666666668, '3': 8.0},
 '128': {'1': 10.833333333333332,
  '2': 8.166666666666668,
  '3': 7.833333333333334},
 '129': {'1': 11.0},
 '130': {'1': 11.166666666666668},
 '131': {'1': 10.833333333333332},
 '132': {'1': 10.833333333333332, '2': 8.0, '3': 8.0},
 '133': {'1': 

In [ ]:
aggdict = {l: lambda n: np.nanmean(n) for l in lines}

In [ ]:
full_grid = full_grid.reset_index().drop('dir',axis=1).groupby('stop').agg(aggdict)
full_grid.to_csv('full_grid_node_delays.csv')

In [ ]:
tph_grid = full_grid.copy()
tph_grid = tph_grid.map(lambda a: 0.0)

In [ ]:
tph_grid = full_grid.copy()
tph_grid = tph_grid.map(lambda a: 0.0)
for stop,tphdata in tphdict.items():
	for line, tph in tphdata.items():
		tph_grid.loc[stop, line] = tph

In [ ]:
tph_grid.to_csv('data/tph_grid.csv')

## get rush hour

In [ ]:
def rush_tph():
	expected_trips = pd.read_csv('gtfs_subway/trips.txt')
	expected_times = pd.read_csv('gtfs_subway/stop_times.txt')
	expected_trips = expected_trips[expected_trips.service_id == 'Weekday']
	expected_trips = expected_trips[expected_trips.route_id != 'SI']
	expected_times = expected_times[expected_times.trip_id.isin(expected_trips.trip_id)]
	routedict = dict(zip(expected_trips.trip_id, expected_trips.route_id))
	dirdict = dict(zip(expected_trips.trip_id, expected_trips.direction_id))
	expected_times['dir'] = expected_times.trip_id.map(lambda id: dirdict.get(id))
	expected_times['route_id'] = expected_times.trip_id.map(lambda id: routedict.get(id))
	expected_times['hour'] = expected_times.arrival_time.map(lambda h: float(h[:4].replace(':','.')))
	expected_times_morning = expected_times[expected_times.hour.between(6,7,'both')]
	expected_times_evening = expected_times[expected_times.hour.between(17,18,'both')]
	expected_times = pd.concat([expected_times_morning, expected_times_evening])
	tph = expected_times.copy()[['stop_id','dir','route_id']]
	tph.dir = tph.dir.map(lambda d: "NS"[d])
	tph = tph.groupby(['stop_id','dir']).agg({'route_id':lambda w: w.to_list()})

	tph['tph'] = tph.route_id.map(lambda w: (pd.value_counts(w)/2).to_dict()).values
	tphavg = tph.drop('route_id', axis=1)
	tphavg['route'] = tph.tph.map(lambda w: list(w.keys()))
	tphavg['tphs'] = tph.tph.map(lambda w: list(w.values()))
	tphavg.drop('tph',axis=1,inplace=True)
	tphavg = tphavg.reset_index().explode(['route','tphs'])
	tphavg.tphs = tphavg.tphs.astype(float)
	tphavg = tphavg.groupby(['stop_id','route']).agg({'tphs':'mean'})
	tphavg.reset_index(inplace=True)
	tphavg.stop_id = tphavg.stop_id.astype(str)
	tphavg = tphavg.sort_values('stop_id').groupby('stop_id').agg(lambda x: x.to_list())
	tphavg.reset_index(inplace=True)
	tphdict = dict(zip(tphavg.stop_id, [dict(zip(a,b)) for a,b in zip(tphavg.route, tphavg.tphs)]))
	return tphdict

In [ ]:
rush_tph()

{'101': {'1': 11.5},
 '103': {'1': 12.0},
 '104': {'1': 12.5},
 '106': {'1': 12.0},
 '107': {'1': 12.5},
 '108': {'1': 12.75},
 '109': {'1': 12.0},
 '110': {'1': 12.0},
 '111': {'1': 12.5},
 '112': {'1': 12.0},
 '113': {'1': 12.5},
 '114': {'1': 12.0},
 '115': {'1': 12.75},
 '116': {'1': 12.5},
 '117': {'1': 12.5},
 '118': {'1': 12.75},
 '119': {'1': 12.25},
 '120': {'1': 13.0, '2': 9.5, '3': 7.5},
 '121': {'1': 12.5, '2': 0.5},
 '122': {'1': 12.25, '2': 0.5},
 '123': {'2': 9.5, '3': 7.5, '1': 12.5},
 '124': {'1': 12.25},
 '125': {'1': 12.5},
 '126': {'1': 12.75},
 '127': {'1': 12.5, '2': 9.0, '3': 7.5},
 '128': {'1': 12.75, '2': 9.25, '3': 7.25},
 '129': {'1': 13.0},
 '130': {'1': 13.0},
 '131': {'1': 12.5},
 '132': {'3': 7.5, '1': 12.75, '2': 9.25},
 '133': {'1': 12.75},
 '134': {'1': 12.5},
 '135': {'1': 13.0},
 '136': {'1': 13.0},
 '137': {'1': 12.5, '2': 9.25, '3': 7.25},
 '138': {'1': 12.75},
 '139': {'1': 13.25},
 '142': {'1': 12.75},
 '201': {'2': 8.5},
 '204': {'5': 2.5, '2': 

In [ ]:
rush_tph_grid = full_grid.copy().map(lambda a: 0.0)
for stop,tphdata in rush_tph().items():
	for line, tph in tphdata.items():
		rush_tph_grid.loc[stop, line] = tph

In [ ]:
rush_tph_grid.to_csv('data/rush_tph.csv')

In [ ]:
def rush_deltas_morning(day):
	mtatime = [6*60*100, 7*60*100]
	expected_trips = pd.read_csv('gtfs_subway/trips.txt')
	actual_trips = pd.read_csv(f"sept27data/subwaydatanyc_2025-09-{day}_trips.csv")

	expected_times = pd.read_csv('gtfs_subway/stop_times.txt')
	actual_times = pd.read_csv(f"sept27data/subwaydatanyc_2025-09-{day}_stop_times.csv")

	def format_time(t):
		'''Helper to format MTA timestamps that go over 24 hours due to delays'''
		rd = [int(n) for n in t.split(':')]
		d = 0
		if rd[0] >= 24:
			rd[0] = rd[0] % 24
			d = 1
		s = ':'.join([str(k) for k in rd])
		datestr = datetime.strptime(f"9/{day+d}/2025 {s}", "%m/%d/%Y %H:%M:%S")
		return datestr


	actual_trips['init_time'] = actual_trips.trip_id.map(lambda s: int(s.split('_')[0]))
	actual_trips = actual_trips[actual_trips.init_time.between(mtatime[0],mtatime[1])]
	actual_trips = actual_trips[actual_trips.route_id != 'SI']
	actual_trips.sort_values('init_time', inplace=True)
	actual_trips['short_id'] = actual_trips.trip_id.map(lambda s: ''.join(re.split(r"(\.{1,2}[SN])",s)[:-1]))
	expected_trips = expected_trips[~expected_trips.route_id.isin(['SI','SS'])]
	expected_trips = expected_trips[expected_trips.service_id == 'Weekday']
	expected_trips['short_id'] = expected_trips.trip_id.map(lambda s: '_'.join(s.split('_')[-2:]))
	expected_trips['init_time'] = expected_trips.trip_id.map(lambda s: int(s.split('_')[-2]))
	expected_trips = expected_trips[expected_trips.init_time.between(mtatime[0],mtatime[1])]
	expected_trips.short_id = expected_trips.short_id.map(lambda s: s[:-3])
	expected_trips = expected_trips.astype(str)
	expected_trips = expected_trips[['route_id','short_id', 'trip_id']]
	expected_trips.columns = ['route_id', 'short_id', 'full_trip_id']
	actual_trips = actual_trips[['trip_uid','route_id','trip_id','direction_id','start_time','short_id','init_time']]
	merged = pd.merge(actual_trips, expected_trips, on='short_id')
	merged['unique'] = merged.trip_id.map(lambda s: merged.trip_id.value_counts(sort=False).to_dict()[s])
	merged['route_id'] = list(zip(merged['route_id_x'], merged['route_id_y']))
	merged.route_id = merged.route_id.map(lambda x: x[0] if len(np.unique_values(list(x)))==1 else x)
	idkey = {v: k for k,v in zip(merged.trip_uid, merged.full_trip_id)}
	expected_times = expected_times[~expected_times.trip_id.str.startswith('SIR-')]
	expected_times['id_key'] =expected_times.trip_id.map(lambda s: '_'.join(s.split('-')[-1].split('_')[1:]))
	expected_times['key'] = expected_times.trip_id.map(lambda k: idkey.get(k))+expected_times.stop_id.astype(str)
	expected_times = expected_times[~expected_times.key.isna()]
	expected_times.sort_values('key')
	expected_times.columns = ['full_trip_id', 'stop_id','expected_arrival','departure','seq','short_id','key']
	real_time = actual_times.copy()
	real_time = real_time[real_time['trip_uid'].isin(merged.trip_uid)]
	real_time.stop_id = real_time.stop_id.map(lambda a: str(a[:-1]))
	real_time = real_time[~(real_time.arrival_time.isna() & real_time.departure_time.isna())]
	real_time['key'] = [a+b for a,b in zip(real_time.trip_uid, real_time.stop_id)]
	real_time.arrival_time = [int(a[np.nanargmax(a)]) for a in zip(real_time.arrival_time, real_time.departure_time)]
	real_time['init_time'] = real_time.trip_uid.map(lambda s: int(s.split('_')[0]))
	real_time.arrival_time = real_time.arrival_time.map(lambda t: datetime.fromtimestamp(t).time())
	real_time.sort_values('key')
	ddict = dict(zip(expected_trips.full_trip_id.values,expected_trips.short_id.map(lambda n: n[-1])))
	iddict = dict(zip(expected_trips.full_trip_id, expected_trips.route_id))
	expected_times['route'] = expected_times.full_trip_id.map(lambda x: iddict.get(x))
	expected_times['direction'] = expected_times.full_trip_id.map(lambda x: ddict.get(x))
	full_merge =pd.merge(real_time, expected_times, on='key')
	full_merge = full_merge[full_merge.expected_arrival.notna()]
	full_merge = full_merge[['stop_id_x','arrival_time','expected_arrival','seq', 'route','direction']]
	full_merge.drop('seq', axis=1,inplace=True)
	full_merge['delta']=[(format_time(a)-format_time(b)).total_seconds() for a,b in zip(full_merge.arrival_time.astype(str), full_merge.expected_arrival.astype(str))]
	deltas = full_merge[['stop_id_x','route','direction','delta']]
	return deltas

In [ ]:
def rush_deltas_evening(day):
	mtatime = [17*60*100, 18*60*100]
	expected_trips = pd.read_csv('gtfs_subway/trips.txt')
	actual_trips = pd.read_csv(f"sept27data/subwaydatanyc_2025-09-{day}_trips.csv")

	expected_times = pd.read_csv('gtfs_subway/stop_times.txt')
	actual_times = pd.read_csv(f"sept27data/subwaydatanyc_2025-09-{day}_stop_times.csv")

	def format_time(t):
		'''Helper to format MTA timestamps that go over 24 hours due to delays'''
		rd = [int(n) for n in t.split(':')]
		d = 0
		if rd[0] >= 24:
			rd[0] = rd[0] % 24
			d = 1
		s = ':'.join([str(k) for k in rd])
		datestr = datetime.strptime(f"9/{day+d}/2025 {s}", "%m/%d/%Y %H:%M:%S")
		return datestr


	actual_trips['init_time'] = actual_trips.trip_id.map(lambda s: int(s.split('_')[0]))
	actual_trips = actual_trips[actual_trips.init_time.between(mtatime[0],mtatime[1])]
	actual_trips = actual_trips[actual_trips.route_id != 'SI']
	actual_trips.sort_values('init_time', inplace=True)
	actual_trips['short_id'] = actual_trips.trip_id.map(lambda s: ''.join(re.split(r"(\.{1,2}[SN])",s)[:-1]))
	expected_trips = expected_trips[~expected_trips.route_id.isin(['SI','SS'])]
	expected_trips = expected_trips[expected_trips.service_id == 'Weekday']
	expected_trips['short_id'] = expected_trips.trip_id.map(lambda s: '_'.join(s.split('_')[-2:]))
	expected_trips['init_time'] = expected_trips.trip_id.map(lambda s: int(s.split('_')[-2]))
	expected_trips = expected_trips[expected_trips.init_time.between(mtatime[0],mtatime[1])]
	expected_trips.short_id = expected_trips.short_id.map(lambda s: s[:-3])
	expected_trips = expected_trips.astype(str)
	expected_trips = expected_trips[['route_id','short_id', 'trip_id']]
	expected_trips.columns = ['route_id', 'short_id', 'full_trip_id']
	actual_trips = actual_trips[['trip_uid','route_id','trip_id','direction_id','start_time','short_id','init_time']]
	merged = pd.merge(actual_trips, expected_trips, on='short_id')
	merged['unique'] = merged.trip_id.map(lambda s: merged.trip_id.value_counts(sort=False).to_dict()[s])
	merged['route_id'] = list(zip(merged['route_id_x'], merged['route_id_y']))
	merged.route_id = merged.route_id.map(lambda x: x[0] if len(np.unique_values(list(x)))==1 else x)
	idkey = {v: k for k,v in zip(merged.trip_uid, merged.full_trip_id)}
	expected_times = expected_times[~expected_times.trip_id.str.startswith('SIR-')]
	expected_times['id_key'] =expected_times.trip_id.map(lambda s: '_'.join(s.split('-')[-1].split('_')[1:]))
	expected_times['key'] = expected_times.trip_id.map(lambda k: idkey.get(k))+expected_times.stop_id.astype(str)
	expected_times = expected_times[~expected_times.key.isna()]
	expected_times.sort_values('key')
	expected_times.columns = ['full_trip_id', 'stop_id','expected_arrival','departure','seq','short_id','key']
	real_time = actual_times.copy()
	real_time = real_time[real_time['trip_uid'].isin(merged.trip_uid)]
	real_time.stop_id = real_time.stop_id.map(lambda a: str(a[:-1]))
	real_time = real_time[~(real_time.arrival_time.isna() & real_time.departure_time.isna())]
	real_time['key'] = [a+b for a,b in zip(real_time.trip_uid, real_time.stop_id)]
	real_time.arrival_time = [int(a[np.nanargmax(a)]) for a in zip(real_time.arrival_time, real_time.departure_time)]
	real_time['init_time'] = real_time.trip_uid.map(lambda s: int(s.split('_')[0]))
	real_time.arrival_time = real_time.arrival_time.map(lambda t: datetime.fromtimestamp(t).time())
	real_time.sort_values('key')
	ddict = dict(zip(expected_trips.full_trip_id.values,expected_trips.short_id.map(lambda n: n[-1])))
	iddict = dict(zip(expected_trips.full_trip_id, expected_trips.route_id))
	expected_times['route'] = expected_times.full_trip_id.map(lambda x: iddict.get(x))
	expected_times['direction'] = expected_times.full_trip_id.map(lambda x: ddict.get(x))
	full_merge =pd.merge(real_time, expected_times, on='key')
	full_merge = full_merge[full_merge.expected_arrival.notna()]
	full_merge = full_merge[['stop_id_x','arrival_time','expected_arrival','seq', 'route','direction']]
	full_merge.drop('seq', axis=1,inplace=True)
	full_merge['delta']=[(format_time(a)-format_time(b)).total_seconds() for a,b in zip(full_merge.arrival_time.astype(str), full_merge.expected_arrival.astype(str))]
	deltas = full_merge[['stop_id_x','route','direction','delta']]
	return deltas

In [ ]:
r22 = pd.concat([rush_deltas_morning(22), rush_deltas_evening(22)])
r23 = pd.concat([rush_deltas_morning(23), rush_deltas_evening(23)])
r24 = pd.concat([rush_deltas_morning(24), rush_deltas_evening(24)])
r25 = pd.concat([rush_deltas_morning(25), rush_deltas_evening(25)])
r26 = pd.concat([rush_deltas_morning(26), rush_deltas_evening(26)])

In [ ]:
sep27rush = pd.concat([r22,r23,r24,r25,r26]).reset_index(drop=True)
sep27rush.columns = ['stop','line','dir','delta']
sep27ravg = sep27rush.groupby(["stop",'dir','line']).agg({'delta':'mean'})
deltadictr=dict(zip(sep27ravg.index.values, sep27ravg.delta))
deltadictr

{('101', 'N', '1'): 148.66326530612244,
 ('101', 'S', '1'): 0.0,
 ('103', 'N', '1'): 163.80612244897958,
 ('103', 'S', '1'): 1.4259259259259258,
 ('104', 'N', '1'): 146.16326530612244,
 ('104', 'S', '1'): -9.194444444444445,
 ('106', 'N', '1'): 146.1734693877551,
 ('106', 'S', '1'): -14.87037037037037,
 ('107', 'N', '1'): 143.8673469387755,
 ('107', 'S', '1'): -11.481481481481481,
 ('108', 'N', '1'): 152.81632653061226,
 ('108', 'S', '1'): 7.703703703703703,
 ('109', 'N', '1'): 139.87755102040816,
 ('109', 'S', '1'): 6.518518518518518,
 ('110', 'N', '1'): 142.28571428571428,
 ('110', 'S', '1'): 5.953703703703703,
 ('111', 'N', '1'): 140.9795918367347,
 ('111', 'S', '1'): 9.425925925925926,
 ('112', 'N', '1'): 143.69387755102042,
 ('112', 'S', '1'): 16.00925925925926,
 ('113', 'N', '1'): 155.69387755102042,
 ('113', 'S', '1'): 25.37037037037037,
 ('114', 'N', '1'): 158.30612244897958,
 ('114', 'S', '1'): 13.962962962962964,
 ('115', 'N', '1'): 147.57142857142858,
 ('115', 'S', '1'): 7.7

In [ ]:
all_stops = get_stops()[:-21]
lines = sorted(['A','C','E','B','D','F','FX','M','G','J','Z','L','N','Q','R','W','FS','GS','H','1','2','3','4','5','6','6X','7','7X'])
mtidxr = pd.MultiIndex.from_tuples(list(product(all_stops, ['N','S'])), names=['stop','dir'])
full_gridr = pd.DataFrame(index=mtidxr, columns=lines)
for k,v in deltadictr.items():
	full_gridr.at[(k[0],k[1]),k[2]] = v

In [ ]:
full_gridr

1    2    3    4    5    6   6X    7   7X    A  ...   GS  \
stop dir                                                           ...        
101  N    148.663265  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
     S           0.0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
103  N    163.806122  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
     S      1.425926  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
104  N    146.163265  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
...              ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   
S01  S           NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
S03  N           NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
     S           NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
S04  N           NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
     S           NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   

            H    J    L    M    N    Q    R    W    Z  
stop dir                                               
101  N    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
     S    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
103  N    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
     S    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
104  N    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
...       ...  ...  ...  ...  ...  ...  ...  ...  ...  
S01  S    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
S03  N    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
     S    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
S04  N    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
     S    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  

[950 rows x 28 columns]

In [ ]:
full_gridr[full_gridr.isna().all(1)]

,,1,2,3,4,5,6,6X,7,7X,A,...,GS,H,J,L,M,N,Q,R,W,Z
stop,dir,,,,,,,,,,,,,,,,,,,,,
711,S,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
713,S,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
H01,S,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
full_gridr.loc[['710','711','712','713','714']]

1    2    3    4    5    6   6X          7   7X    A  ...   GS  \
stop dir                                                          ...        
710  N    NaN  NaN  NaN  NaN  NaN  NaN  NaN  69.061224  NaN  NaN  ...  NaN   
     S    NaN  NaN  NaN  NaN  NaN  NaN  NaN  36.918478  NaN  NaN  ...  NaN   
711  N    NaN  NaN  NaN  NaN  NaN  NaN  NaN  37.435374  NaN  NaN  ...  NaN   
     S    NaN  NaN  NaN  NaN  NaN  NaN  NaN        NaN  NaN  NaN  ...  NaN   
712  N    NaN  NaN  NaN  NaN  NaN  NaN  NaN  69.877551  NaN  NaN  ...  NaN   
     S    NaN  NaN  NaN  NaN  NaN  NaN  NaN   5.195652  NaN  NaN  ...  NaN   
713  N    NaN  NaN  NaN  NaN  NaN  NaN  NaN  -0.823129  NaN  NaN  ...  NaN   
     S    NaN  NaN  NaN  NaN  NaN  NaN  NaN        NaN  NaN  NaN  ...  NaN   
714  N    NaN  NaN  NaN  NaN  NaN  NaN  NaN -16.054422  NaN  NaN  ...  NaN   
     S    NaN  NaN  NaN  NaN  NaN  NaN  NaN   2.804348  NaN  NaN  ...  NaN   

            H    J    L    M    N    Q    R    W    Z  
stop dir                                               
710  N    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
     S    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
711  N    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
     S    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
712  N    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
     S    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
713  N    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
     S    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
714  N    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
     S    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  

[10 rows x 28 columns]

In [ ]:
class BreakPoint(Exception):
	pass

In [ ]:
## somehow the 7X is not showing up on this map, no idea why

In [ ]:
## we can extrapolate 711 and 713 Southbound since they dont share tracks and only use the 7, so the average delay from previous and following stops will work here
full_gridr.at[('711','S'),'7'] = (full_gridr.loc['710','S']['7']+full_gridr.loc['712','S']['7'])/2
full_gridr.at[('713','S'),'7'] = (full_gridr.loc['712','S']['7']+full_gridr.loc['714','S']['7'])/2

## aqueduct racetrack is a oneway stop so we can just assume the delay would be the same
full_gridr.at[('H01','S'),'A'] = full_gridr.at[('H01','N'),'A']
full_gridr[full_gridr.isna().all(1)]
full_gridr_md = full_gridr.copy()
aggstops = {k: lambda x: np.nanmean(x, axis=0, dtype=np.float64) for k in lines}
aggstops['stop'] = 'first'
full_gridr_md_avg = full_gridr_md.reset_index().drop('dir', axis=1).groupby('stop').agg(aggstops)
delay_timesr = full_gridr_md_avg.drop('stop',axis=1).apply(lambda x: np.nanmean(x),axis=1)
np.round(delay_timesr.values.clip(0)/60).clip(0,5)
rushmindf = pd.DataFrame(delay_timesr, columns=['seconds'])
rushmindf['mins'] = np.round(delay_timesr.values.clip(0)/60).clip(0,5)
rushmindf.mins.to_csv('data/delay_minutes_rush.csv')